## Life Expectancy Analysis
Analyzing the WHO Life Expectancy dataset to explore which health, economic and social factors are most associated with life expectancy across countries.

Goal: Build a regression model to predict life ecpectancy from these factors, and use hypothesis testing to determine whether specific factors have have a statistically significant  relationship with it

Dataset source: Kaggle - "Life Expectancy (WHO)" by Kumar Rajarshi

In [ ]:
# importing the relevant libraries
import pandas as pd, numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt

In [ ]:
## Step 1: Load and Clean Data
#Load the dataset and remove whitespace from column names to prevent errors when referencing columns later 

In [ ]:
df = pd.read_csv('Life Expectancy Data.csv')
df.columns = df.columns.str.strip()
df.head()

In [ ]:
## Step 2: CHeck  for missing values
#Before cleaning, check which columns have missing values and how much

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
## Step 3: Handle missing values
# -fill numeric columns with median value per country to avoid distorting values with unrelated countries' data(not mean because there is high likelihood of extreme outliers for each country)
# -drop rows where life expectancy or adult mortality is missing, since these are small in number and unreliable to impute for a regression target.

In [ ]:
# dealing with the numeric columns
cols_to_fill = ['GDP', 'Alcohol', 'Total expenditure', 'Schooling', 'Income composition of resources', 'BMI', 'thinness  1-19 years', 'thinness 5-9 years', 'Hepatitis B', 'Polio', 'Diphtheria', 'Population']
for col in cols_to_fill:
    df[col] = df.groupby('Country')[col].transform(lambda x: x.fillna(x.median()))

# dealing with rows where life expectancy or adult mortality is missing
df = df.dropna(subset=['Life expectancy', 'Adult Mortality']).copy()

# confirm remaining missing values
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# filling remaining missing values with the overall median
for col in cols_to_fill:
    df[col] = df[col].fillna(df[col].median())
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df[df['Schooling'] < 1]

In [ ]:
## Step 3b: Fixing Hidden Missing Values
# Schooling and Income composition of resources had 0.0 values in 26 rows that are implausible as real measurements. These were converted to NaN and filled using the same
# country-median approach as the rest of Step 3

In [ ]:
df['Schooling'] = df['Schooling'].replace(0, np.nan)
df['Income composition of resources'] = df['Income composition of resources'].replace(0, np.nan)

df['Schooling'] = df.groupby('Country')['Schooling'].transform(lambda x: x.fillna(x.median()))
df['Income composition of resources'] = df.groupby('Country')['Income composition of resources'].transform(lambda x: x.fillna(x.median()))

df['Schooling'] = df['Schooling'].fillna(df['Schooling'].median())
df['Income composition of resources'] = df['Income composition of resources'].fillna(df['Income composition of resources'].median())

In [ ]:
df[df['Schooling'] < 1]

In [ ]:
## Step 4: Hypothesis Testing
# Testing whether life expectancy differs significantly between Developed and Developing countries using an independent t-test.
# null hypothesis (Ho):      There is no significant difference in mean life between the two groups
# alternate hypothesis (H1): There is a significant difference in mean life between the two groups.

In [ ]:
#importing a relevant library
from scipy import stats
developed = df[df['Status'] == 'Developed']['Life expectancy']
developing = df[df['Status'] == 'Developing']['Life expectancy']

t_stat, p_value = stats.ttest_ind(developed, developing)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.10f}")

if p_value < 0.05:
    print("Result: Reject Ho - statistically significant difference.")
else:
    print("Result: Fail to reject Ho - no significant difference.")

In [ ]:
## Interpretation
# The near-zero p-value means that if Ho were true, seeing such a large difference between the life expectancy of developed and developing countries would be almost inpossible.
# We therefore rejct Ho in favor of H1, implying that development status has a statistically significant relationship with life expectancy

## Building the Regression Model

In [ ]:
## Step 5: Feature Selection
# Check correlation of each numeric feature with life expectancy before building the regression model.

In [ ]:
numeric_df = df.select_dtypes(include='number')
correlations = numeric_df.corr()['Life expectancy'].sort_values(ascending=False)
print(correlations)

In [ ]:
## Step 6: Feature Selection for Regression
# Selected features with strong correlation to life expectancy, avoiding redundant pairs (e.g. thinness 1-19 vs 5-9, infant vs under 5 deaths) to reduce multicollinearity

In [ ]:
features = ['Schooling', 'Income composition of resources', 'Adult Mortality', 'HIV/AIDS', 'BMI', 'GDP', 'Diphtheria', 
            'thinness 5-9 years', 'under-five deaths']
x = df[features]
y = df['Life expectancy']

print(x.shape, y.shape)

In [ ]:
## Step 7: Train/Test Split
# Split data into training (80%) and testing (20%) sets to evaluate the model on unseen data

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

In [ ]:
## Step 8: Train the Regression Model
# Fit a linear regression model on the training data.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x_train, y_train)

print("Model trained.")
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

In [ ]:
## Step 9: Evaluate Model Performance
# Test the model on unseen data using R^2 and RMSE

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

y_pred = model.predict(x_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R^2 Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

In [ ]:
## Step 10: Visualizations
# Visualize relationships in the data and examine where the model performs worst.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap of all Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize = (8, 6))
sns.scatterplot(data = df, x = 'Schooling', y = 'Life expectancy', hue = 'Status', alpha = 0.6)
plt.title('Schooling vs Life Expectancy')
plt.xlabel('Average Years of Schooling')
plt.ylabel('Life Expectancy')
plt.show()

In [ ]:
## Step 11: Data QUality Note
# During an earlier pass of this analysis, the Schooling vs Lufe Expectancy scatter plot showed a vertical cluster of points at the Schooling = 0, accounting for an
# implausibly wide range of of Life Expectancy values. Investigating this revealed 26 rows where Schooling and Income composition were both recorded as 0.0, likely 
# missing HDI data misrepresented as genuine zero values. This was corrected the cleaning stage (see Step 3b), so the scatter plot shown above represents the corrected
# data, and no longer displays the artefact described here.

## Conclusion

### Hypothesis Test
There is a statistically significant difference (p < 0.001) in life expectancy between Developed and Developing countries, confirming that development status meaningfully affects health outcomes

### Strongest Predictors
Schooling, Income composition of resources, and Adult Mortality showed the strongest relationships with life expectancy - education and economic development appear to be closely tied to how long people live.

### Model Performance
A linear regression using 9 features explained about 83% of the variation in life expectancy (R^2 = 0.8325), with an average prediction error of about 3.8 years (RMSE = 3.8065)

### Notable Observation
Alcohol consumption showed a moderate positive correlation with life expectancy, though this is likely to be a confounding effect of wealth (wealthier countries drink more and also have better healthcare), not evidence that alcohol itself improves health outcomes, leading to higher life expectancy.

### Limitations
Remaining missing data was imputed using country-level medians, which may not perfectly reflect true values. A linear model also assumes straight-line relationships, whicih may miss more complex patterns a non-linear model could capture.